# E5  :  Fine-tuning MTEB French

**Objectif :** Fine-tuner `intfloat/multilingual-e5-base` sur un mélange français retrieval + STS + paraphrases + NLI, avec replay multilingue.  
**Hardware cible :** RTX 3070 (8 GB VRAM)  
**Approche :** Sentence Transformers + PEFT LoRA, split anti-fuite, pertes multi-tâches, sélection retrieval/STS et contrôles gradients/poids/rechargement.  
**Références :** [E5 multilingual](https://huggingface.co/intfloat/multilingual-e5-base) · [Training multi-dataset Sentence Transformers](https://sbert.net/docs/sentence_transformer/training_overview.html#multi-dataset-training)

## 0. Imports & vérifications

In [1]:
# Cellule 0 — redémarrer le kernel puis exécuter ceci en premier

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", device)

if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("vrAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

torch: 2.12.1+cu130
cuda: True
device: cuda
gpu: NVIDIA GeForce RTX 3070
vrAM GB: 8.0


In [2]:

# Cellule 1 — imports

from datasets import Dataset, concatenate_datasets, load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import (
    CachedMultipleNegativesRankingLoss,
    CoSENTLoss,
    CosineSimilarityLoss,
    EmbedDistillLoss,
    MarginMSELoss,
)
from sentence_transformers.sentence_transformer.evaluation import (
    EmbeddingSimilarityEvaluator,
    InformationRetrievalEvaluator,
    SimilarityFunction,
)
from sentence_transformers.base.evaluation import BaseEvaluator
from sentence_transformers.base.sampler import BatchSamplers, MultiDatasetBatchSamplers
from sentence_transformers.util import cos_sim
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import TrainerCallback

print("Imports OK")


W0709 15:19:18.355000 3940 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0709 15:19:18.427000 3940 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Imports OK


In [3]:

# Cellule 2 — configuration SAFE v2

MODEL_NAME = "intfloat/multilingual-e5-base"

# Nouveau dossier: conservative v3. Le run safe précédent dérivait encore trop.
OUTPUT_DIR = "./e5-fr-conservative-lora"
MAX_SEQ_LENGTH = 256

# Objectif de sélection. Options:
# - "global_safe"    : cherche un gain équilibré retrieval + STS + rétention baseline.
# - "sts_safe"       : cherche surtout STS, mais interdit la régression retrieval/rétention.
# - "retrieval_safe" : cherche surtout retrieval, mais interdit la régression STS/rétention.
SAFE_OBJECTIVE = "global_safe"

# Conservative v3: bouger très peu le modèle et laisser la distillation dominer.
# Le run précédent a amélioré STS mais a déplacé l'espace embedding global.
TRAIN_SIZE = 12_000
MIRACL_MARGIN_SIZE = 16_000
NLI_TRAIN_SIZE = 3_000
STS_TRAIN_SIZE = 2_000
PARAPHRASE_TRAIN_SIZE = 500
REPLAY_PER_LANGUAGE = 6_000
REPLAY_CONFIGS = ("en-fr", "en-de", "en-es", "en-ar")
BASELINE_DISTILL_SIZE = 48_000
DISTILL_BATCH_SIZE = 128

EVAL_QUERY_SIZE = 300
EVAL_NEGATIVES_PER_QUERY = 16
STS_EVAL_SIZE = 2_000
RETENTION_PROBE_SIZE = 512

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64

NUM_EPOCHS = 1
LEARNING_RATE = 2e-6
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
EVAL_STEPS = 100

# Garde-fous: un checkpoint peut gagner, mais pas au prix d'un drift global.
MAX_RETRIEVAL_DROP = 0.002
MAX_STS_DROP = 0.003
MIN_RETENTION_COSINE = 0.985
MIN_PAIRWISE_RETENTION = 0.985
MIN_ROBUST_DELTA = 0.0002
REGRESSION_PENALTY = 10.0
HARD_STOP_RETENTION_COSINE = 0.965
HARD_STOP_PAIRWISE_RETENTION = 0.930
MAX_LORA_B_ABS = 0.006
MAX_BAD_EVALS = 2
LOSS_WEIGHTS = {
    "miracl": 0.08,
    "miracl_margin": 0.35,
    "fr_sts": 0.02,
    "fr_paraphrase": 0.02,
    "fr_nli": 0.005,
    "multilingual_replay": 0.12,
    "baseline_distill": 8.0,
}
OBJECTIVE_WEIGHTS = {
    "retrieval": 0.45,
    "sts": 0.25,
    "retention": 0.30,
}

assert TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS == 64
assert SAFE_OBJECTIVE in {"global_safe", "sts_safe", "retrieval_safe"}

print({
    "objective": SAFE_OBJECTIVE,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "miracl_rank_size": TRAIN_SIZE,
    "miracl_margin_size": MIRACL_MARGIN_SIZE,
    "nli_train_size": NLI_TRAIN_SIZE,
    "sts_train_size": STS_TRAIN_SIZE,
    "paraphrase_train_size": PARAPHRASE_TRAIN_SIZE,
    "multilingual_replay_size": REPLAY_PER_LANGUAGE * len(REPLAY_CONFIGS),
    "baseline_distill_size": BASELINE_DISTILL_SIZE,
    "heldout_queries": EVAL_QUERY_SIZE,
    "epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,
    "loss_weights": LOSS_WEIGHTS,
})


{'objective': 'global_safe', 'effective_batch_size': 64, 'miracl_rank_size': 12000, 'miracl_margin_size': 16000, 'nli_train_size': 3000, 'sts_train_size': 2000, 'paraphrase_train_size': 500, 'multilingual_replay_size': 24000, 'baseline_distill_size': 48000, 'heldout_queries': 300, 'epochs': 1, 'lr': 2e-06, 'loss_weights': {'miracl': 0.08, 'miracl_margin': 0.35, 'fr_sts': 0.02, 'fr_paraphrase': 0.02, 'fr_nli': 0.005, 'multilingual_replay': 0.12, 'baseline_distill': 8.0}}


In [4]:
# Cellule 3 — chargement modèle baseline

model = SentenceTransformer(
    MODEL_NAME,
    device=device,
)

model.max_seq_length = MAX_SEQ_LENGTH

print("embedding dim:", model.get_embedding_dimension())
print("max_seq_length:", model.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
f:\codeAI\llm\.venv\lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


embedding dim: 768
max_seq_length: 256


In [5]:
# Cellule 4 — test obligatoire de convention E5

queries = [
    "query: Quelle est la capitale de la France ?",
    "query: Quel animal miaule ?",
]

passages = [
    "passage: Paris est la capitale de la France.",
    "passage: Le chat est un animal domestique qui miaule.",
    "passage: Le moteur d'une voiture transforme de l'énergie en mouvement.",
]

q_emb = model.encode(queries, normalize_embeddings=True)
p_emb = model.encode(passages, normalize_embeddings=True)

scores = q_emb @ p_emb.T

print(scores)

assert scores[0, 0] > scores[0, 1]
assert scores[1, 1] > scores[1, 2]

print("Convention query/passage OK")

[[0.8838406  0.70628965 0.7135833 ]
 [0.7198016  0.82489645 0.73087937]]
Convention query/passage OK


In [6]:
# Cellule 5 — chargement des sources (uniquement leurs splits train/validation)

ds = load_dataset(
    "sentence-transformers/miracl", "fr-triplet-all", split="train"
)
fr_sts_splits = load_dataset("CATIE-AQ/frenchSTS")
fr_nli_raw = load_dataset("CATIE-AQ/frenchNLI", split="train")
replay_raw = {
    config: load_dataset(
        "sentence-transformers/parallel-sentences-global-voices",
        config,
        split=f"train[:{REPLAY_PER_LANGUAGE}]",
    )
    for config in REPLAY_CONFIGS
}

print({
    "miracl_rows": len(ds),
    "french_sts_train_rows": len(fr_sts_splits["train"]),
    "french_nli_rows": len(fr_nli_raw),
    "replay_configs": list(replay_raw),
})

{'miracl_rows': 232100, 'french_sts_train_rows': 12227, 'french_nli_rows': 555798, 'replay_configs': ['en-fr', 'en-de', 'en-es', 'en-ar']}


In [7]:

# Cellule 6 — split MIRACL groupé + préparation multi-dataset sans fuite + anti-drift

anchor_column = ds["anchor"]
positive_column = ds["positive"]
negative_column = ds["negative"]
unique_anchors = np.array(sorted(set(anchor_column)), dtype=object)
split_rng = np.random.default_rng(SEED)
split_rng.shuffle(unique_anchors)

assert EVAL_QUERY_SIZE < len(unique_anchors)
eval_anchor_set = set(unique_anchors[:EVAL_QUERY_SIZE].tolist())
eval_indices = [i for i, anchor in enumerate(anchor_column) if anchor in eval_anchor_set]
eval_ds = ds.select(eval_indices)
eval_positive_texts = set(eval_ds["positive"])
heldout_texts = eval_anchor_set | eval_positive_texts

# Aucun positif du holdout ne peut apparaître dans MIRACL train, même comme négatif.
train_candidate_indices = [
    i
    for i, (anchor, positive, negative) in enumerate(
        zip(anchor_column, positive_column, negative_column)
    )
    if anchor not in eval_anchor_set
    and positive not in eval_positive_texts
    and negative not in eval_positive_texts
]
split_rng.shuffle(train_candidate_indices)
assert len(train_candidate_indices) >= TRAIN_SIZE
train_ds = ds.select(train_candidate_indices[:TRAIN_SIZE]).select_columns(
    ["anchor", "positive", "negative"]
)


def remove_heldout_texts(dataset, text_columns, description):
    return dataset.filter(
        lambda row: all(row[column] not in heldout_texts for column in text_columns),
        desc=description,
    )


# STS français scoré -> CoSENT. OpusParcus est exclu pour rester sur le chemin CC-BY.
sts_source = fr_sts_splits["train"].filter(
    lambda row: row["dataset"] != "opusparcus", desc="Filtrage licence STS"
)
sts_source = remove_heldout_texts(
    sts_source, ["sentence1", "sentence2"], "Anti-leak FrenchSTS"
).shuffle(seed=SEED)
sts_train = sts_source.select(range(min(STS_TRAIN_SIZE, len(sts_source)))).select_columns(
    ["sentence1", "sentence2", "score"]
)

# Paires paraphrastiques fortes, mais volume réduit pour éviter de transformer le modèle en pur STS.
paraphrase_source = sts_source.filter(
    lambda row: row["score"] >= 0.80, desc="Sélection paraphrases FR"
).shuffle(seed=SEED + 1)
paraphrase_train = paraphrase_source.select(
    range(min(PARAPHRASE_TRAIN_SIZE, len(paraphrase_source)))
).select_columns(["sentence1", "sentence2"]).rename_columns(
    {"sentence1": "anchor", "sentence2": "positive"}
)

# NLI français: implication=1, neutre=0.5, contradiction=0 pour CoSENT.
nli_source = remove_heldout_texts(
    fr_nli_raw, ["premise", "hypothesis"], "Anti-leak FrenchNLI"
)
nli_per_label = NLI_TRAIN_SIZE // 3
nli_parts = []
for label in (0, 1, 2):
    label_dataset = nli_source.filter(
        lambda row, expected=label: int(row["label"]) == expected,
        desc=f"FrenchNLI label {label}",
    ).shuffle(seed=SEED + label)
    assert len(label_dataset) >= nli_per_label
    nli_parts.append(label_dataset.select(range(nli_per_label)))
nli_source = concatenate_datasets(nli_parts).shuffle(seed=SEED)
nli_train = nli_source.map(
    lambda row: {
        "sentence1": row["premise"],
        "sentence2": row["hypothesis"],
        "score": {0: 1.0, 1: 0.5, 2: 0.0}[int(row["label"])],
    },
    remove_columns=nli_source.column_names,
    desc="Conversion NLI vers similarité",
)

# Replay multilingue plus fort que le run précédent: il limite l'oubli de la géométrie E5 générale.
replay_parts = []
for config, replay_dataset in replay_raw.items():
    replay_dataset = remove_heldout_texts(
        replay_dataset, ["english", "non_english"], f"Anti-leak replay {config}"
    ).select_columns(["english", "non_english"]).rename_columns(
        {"english": "anchor", "non_english": "positive"}
    )
    replay_parts.append(replay_dataset)
replay_train = concatenate_datasets(replay_parts).shuffle(seed=SEED)

# Distillation de marge MIRACL: le student apprend le signal retrieval sans dépasser brutalement le teacher.
miracl_margin_source = train_ds.shuffle(seed=SEED + 7).select(
    range(min(MIRACL_MARGIN_SIZE, len(train_ds)))
)


def add_teacher_margins(dataset, batch_size=DISTILL_BATCH_SIZE):
    queries = ["query: " + text for text in dataset["anchor"]]
    positives = ["passage: " + text for text in dataset["positive"]]
    negatives = ["passage: " + text for text in dataset["negative"]]
    q_emb = model.encode(
        queries, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True
    )
    p_emb = model.encode(
        positives, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True
    )
    n_emb = model.encode(
        negatives, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True
    )
    labels = (q_emb * p_emb).sum(axis=1) - (q_emb * n_emb).sum(axis=1)
    return dataset.add_column("label", labels.astype("float32").tolist())


miracl_margin_train = add_teacher_margins(miracl_margin_source)

# Distillation baseline: le modèle doit conserver les embeddings E5 sur un mélange large de textes.
def prefixed_items_from_sources():
    items = []

    for row in train_ds.select(range(min(len(train_ds), 12_000))):
        items.append(("query", row["anchor"]))
        items.append(("passage", row["positive"]))
        items.append(("passage", row["negative"]))

    for row in sts_train:
        items.append(("query", row["sentence1"]))
        items.append(("query", row["sentence2"]))

    for row in nli_train:
        items.append(("query", row["sentence1"]))
        items.append(("query", row["sentence2"]))

    for row in replay_train.select(range(min(len(replay_train), 12_000))):
        items.append(("query", row["anchor"]))
        items.append(("query", row["positive"]))

    deduped = []
    seen = set()
    for prompt, text in items:
        key = (prompt, text)
        if text in heldout_texts or key in seen:
            continue
        seen.add(key)
        deduped.append(key)
    split_rng.shuffle(deduped)
    return deduped


distill_items = prefixed_items_from_sources()[:BASELINE_DISTILL_SIZE]
assert len(distill_items) >= BASELINE_DISTILL_SIZE // 2
assert not ({text for _, text in distill_items} & heldout_texts)

distill_texts = [f"{prompt}: {text}" for prompt, text in distill_items]
distill_labels = model.encode(
    distill_texts,
    batch_size=DISTILL_BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")
baseline_distill_train = Dataset.from_dict(
    {"text": distill_texts, "label": distill_labels.tolist()}
)

train_datasets = {
    "miracl": train_ds,
    "miracl_margin": miracl_margin_train,
    "fr_sts": sts_train,
    "fr_paraphrase": paraphrase_train,
    "fr_nli": nli_train,
    "multilingual_replay": replay_train,
    "baseline_distill": baseline_distill_train,
}

# Audit strict: aucune requête/aucun positif held-out dans aucune source textuelle de train.
training_text_columns = {
    "miracl": ["anchor", "positive", "negative"],
    "miracl_margin": ["anchor", "positive", "negative"],
    "fr_sts": ["sentence1", "sentence2"],
    "fr_paraphrase": ["anchor", "positive"],
    "fr_nli": ["sentence1", "sentence2"],
    "multilingual_replay": ["anchor", "positive"],
}
all_training_texts = set()
for dataset_name, dataset in train_datasets.items():
    if dataset_name == "baseline_distill":
        continue
    for column in training_text_columns[dataset_name]:
        all_training_texts.update(dataset[column])
all_training_texts.update(text for _, text in distill_items)

text_overlap = all_training_texts & heldout_texts
assert not text_overlap, f"Data leak multi-dataset: {len(text_overlap)} textes"
assert set(train_ds["anchor"]).isdisjoint(eval_anchor_set)

print({name: len(dataset) for name, dataset in train_datasets.items()})
print({
    "eval_unique_queries": len(eval_anchor_set),
    "multi_dataset_text_overlap": len(text_overlap),
    "distill_rows": len(baseline_distill_train),
})
print("Préparation multi-dataset anti-leak + anti-drift OK")


Conversion NLI vers similarité:   0%|          | 0/3000 [00:00<?, ? examples/s]

Anti-leak replay en-fr:   0%|          | 0/6000 [00:00<?, ? examples/s]

Anti-leak replay en-de:   0%|          | 0/6000 [00:00<?, ? examples/s]

Anti-leak replay en-es:   0%|          | 0/6000 [00:00<?, ? examples/s]

Anti-leak replay en-ar:   0%|          | 0/6000 [00:00<?, ? examples/s]

...

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Flattening the indices:   0%|          | 0/12000 [00:00<?, ? examples/s]

Batches:   0%|          | 0/363 [00:00<?, ?it/s]

{'miracl': 12000, 'miracl_margin': 12000, 'fr_sts': 2000, 'fr_paraphrase': 500, 'fr_nli': 3000, 'multilingual_replay': 24000, 'baseline_distill': 46409}
{'eval_unique_queries': 300, 'multi_dataset_text_overlap': 0, 'distill_rows': 46409}
Préparation multi-dataset anti-leak + anti-drift OK


In [8]:

# Cellule 7 — évaluateur robuste: retrieval + STS + préservation géométrique baseline


def build_ir_evaluator(
    dataset,
    training_texts,
    negatives_per_query=16,
    name="miracl-fr-heldout",
):
    grouped = {}
    for row in dataset:
        group = grouped.setdefault(row["anchor"], {"positive": set(), "negative": set()})
        group["positive"].add(row["positive"])
        group["negative"].add(row["negative"])

    queries = {}
    corpus = {}
    relevant_docs = {}
    doc_id_by_text = {}

    def add_passage(text):
        text = "passage: " + text
        if text not in doc_id_by_text:
            doc_id = f"d{len(doc_id_by_text)}"
            doc_id_by_text[text] = doc_id
            corpus[doc_id] = text
        return doc_id_by_text[text]

    skipped_queries = 0
    for anchor in sorted(grouped):
        positives = sorted(grouped[anchor]["positive"])
        negatives = sorted(
            text
            for text in grouped[anchor]["negative"]
            if text not in training_texts and text not in positives
        )
        if not positives or not negatives:
            skipped_queries += 1
            continue

        qid = f"q{len(queries)}"
        queries[qid] = "query: " + anchor
        relevant_docs[qid] = {add_passage(text) for text in positives}

        query_rng = random.Random(f"{SEED}:{anchor}")
        selected_negatives = query_rng.sample(
            negatives, k=min(negatives_per_query, len(negatives))
        )
        for text in selected_negatives:
            add_passage(text)

    assert queries, "Aucune requête held-out évaluable."
    assert not (set(corpus.values()) & {"passage: " + x for x in training_texts})
    print({
        "heldout_queries_evaluated": len(queries),
        "heldout_queries_skipped": skipped_queries,
        "deduplicated_corpus_size": len(corpus),
    })

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        score_functions={"cosine": cos_sim},
        show_progress_bar=True,
    )


retrieval_evaluator = build_ir_evaluator(
    eval_ds,
    training_texts=all_training_texts,
    negatives_per_query=EVAL_NEGATIVES_PER_QUERY,
)

# Validation STS sans OpusParcus et sans aucun texte vu au train.
sts_validation = fr_sts_splits["validation"].filter(
    lambda row: row["dataset"] != "opusparcus"
    and row["sentence1"] not in all_training_texts
    and row["sentence2"] not in all_training_texts,
    desc="STS validation anti-leak",
).shuffle(seed=SEED)
sts_validation = sts_validation.select(range(min(STS_EVAL_SIZE, len(sts_validation))))
assert len(sts_validation) > 0

sts_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=["query: " + text for text in sts_validation["sentence1"]],
    sentences2=["query: " + text for text in sts_validation["sentence2"]],
    scores=sts_validation["score"],
    main_similarity=SimilarityFunction.COSINE,
    name="fr-sts-heldout",
)

# Probe de rétention: textes held-out/validation, jamais utilisés pour l'entraînement.
retention_items = []
for row in sts_validation:
    retention_items.append("query: " + row["sentence1"])
    retention_items.append("query: " + row["sentence2"])
for row in eval_ds:
    retention_items.append("query: " + row["anchor"])
    retention_items.append("passage: " + row["positive"])
    retention_items.append("passage: " + row["negative"])

retention_probe_texts = []
seen_probe_texts = set()
for text in retention_items:
    if text not in seen_probe_texts:
        seen_probe_texts.add(text)
        retention_probe_texts.append(text)
split_rng.shuffle(retention_probe_texts)
retention_probe_texts = retention_probe_texts[:RETENTION_PROBE_SIZE]
assert len(retention_probe_texts) >= min(128, RETENTION_PROBE_SIZE)


def find_metric(metrics, *needles):
    normalized_needles = [needle.lower().replace(" ", "_") for needle in needles]
    for key, value in metrics.items():
        normalized_key = key.lower().replace(" ", "_")
        if all(needle in normalized_key for needle in normalized_needles):
            return float(value), key
    raise KeyError(f"Metric introuvable: {needles}; clés={sorted(metrics)}")


class RobustFrenchHeldoutEvaluator(BaseEvaluator):
    """Score de sélection anti-régression pour éviter le piège STS-only."""

    def __init__(self, retrieval_eval, sts_eval, retention_texts):
        super().__init__()
        self.primary_metric = "robust_score"
        self.greater_is_better = True
        self.retrieval_eval = retrieval_eval
        self.sts_eval = sts_eval
        self.retention_texts = retention_texts
        self.baseline_embeddings = None
        self.baseline_pairwise = None
        self.baseline = None

    @staticmethod
    def _upper_triangle(matrix):
        indices = np.triu_indices(matrix.shape[0], k=1)
        return matrix[indices]

    def _retention_metrics(self, current_model):
        embeddings = current_model.encode(
            self.retention_texts,
            batch_size=EVAL_BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")
        pairwise = self._upper_triangle(embeddings @ embeddings.T)

        if self.baseline_embeddings is None:
            self.baseline_embeddings = embeddings.copy()
            self.baseline_pairwise = pairwise.copy()

        embedding_cosine = float(np.mean(np.sum(embeddings * self.baseline_embeddings, axis=1)))
        pairwise_alignment = float(1.0 - np.mean(np.abs(pairwise - self.baseline_pairwise)))
        retention_score = min(embedding_cosine, pairwise_alignment)
        return embedding_cosine, pairwise_alignment, retention_score

    def _objective(self, retrieval, sts, retention):
        if SAFE_OBJECTIVE == "sts_safe":
            return sts
        if SAFE_OBJECTIVE == "retrieval_safe":
            return retrieval
        return (
            OBJECTIVE_WEIGHTS["retrieval"] * retrieval
            + OBJECTIVE_WEIGHTS["sts"] * sts
            + OBJECTIVE_WEIGHTS["retention"] * retention
        )

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        retrieval_metrics = self.retrieval_eval(model, output_path=output_path, epoch=epoch, steps=steps)
        sts_metrics = self.sts_eval(model, output_path=output_path, epoch=epoch, steps=steps)
        metrics = {**retrieval_metrics, **sts_metrics}

        retrieval, retrieval_key = find_metric(metrics, "miracl-fr-heldout", "ndcg@10")
        try:
            sts, sts_key = find_metric(metrics, "fr-sts-heldout", "spearman", "cosine")
        except KeyError:
            sts, sts_key = find_metric(metrics, "fr-sts-heldout", "pearson", "cosine")

        retention_cosine, pairwise_retention, retention = self._retention_metrics(model)
        objective = self._objective(retrieval, sts, retention)

        if self.baseline is None:
            objective_delta = 0.0
            robust_score = objective
            passes_guardrails = 1.0
            penalty = 0.0
            retrieval_delta = 0.0
            sts_delta = 0.0
        else:
            retrieval_delta = retrieval - self.baseline["retrieval"]
            sts_delta = sts - self.baseline["sts"]
            objective_delta = objective - self.baseline["objective"]
            retrieval_drop = max(0.0, self.baseline["retrieval"] - retrieval - MAX_RETRIEVAL_DROP)
            sts_drop = max(0.0, self.baseline["sts"] - sts - MAX_STS_DROP)
            retention_drop = max(0.0, MIN_RETENTION_COSINE - retention_cosine)
            pairwise_drop = max(0.0, MIN_PAIRWISE_RETENTION - pairwise_retention)
            delta_drop = max(0.0, MIN_ROBUST_DELTA - objective_delta)
            penalty = REGRESSION_PENALTY * (
                retrieval_drop + sts_drop + retention_drop + pairwise_drop + delta_drop
            )
            robust_score = objective_delta - penalty
            passes_guardrails = float(
                retrieval_delta >= -MAX_RETRIEVAL_DROP
                and sts_delta >= -MAX_STS_DROP
                and retention_cosine >= MIN_RETENTION_COSINE
                and pairwise_retention >= MIN_PAIRWISE_RETENTION
                and objective_delta >= MIN_ROBUST_DELTA
            )

        metrics.update({
            "robust_score": float(robust_score),
            "robust_objective_score": float(objective),
            "robust_objective_delta": float(objective_delta),
            "robust_penalty": float(penalty),
            "robust_passes_guardrails": float(passes_guardrails),
            "robust_retrieval_metric": float(retrieval),
            "robust_sts_metric": float(sts),
            "robust_retrieval_delta": float(retrieval_delta),
            "robust_sts_delta": float(sts_delta),
            "robust_retention_cosine": float(retention_cosine),
            "robust_pairwise_retention": float(pairwise_retention),
            "robust_retention_score": float(retention),
            "sequential_score": float(robust_score),
        })
        return metrics

    def set_baseline(self, metrics):
        self.baseline = {
            "retrieval": float(metrics["robust_retrieval_metric"]),
            "sts": float(metrics["robust_sts_metric"]),
            "retention": float(metrics["robust_retention_score"]),
            "objective": float(metrics["robust_objective_score"]),
        }
        print("Baseline robuste figée:", self.baseline)


evaluator = RobustFrenchHeldoutEvaluator(
    retrieval_evaluator,
    sts_evaluator,
    retention_probe_texts,
)
print({"sts_heldout_rows": len(sts_validation), "retention_probe_rows": len(retention_probe_texts)})
print("Evaluateur robuste anti-leak / anti-régression OK")


{'heldout_queries_evaluated': 300, 'heldout_queries_skipped': 0, 'deduplicated_corpus_size': 5418}


STS validation anti-leak:   0%|          | 0/3526 [00:00<?, ? examples/s]

{'sts_heldout_rows': 1552, 'retention_probe_rows': 512}
Evaluateur robuste anti-leak / anti-régression OK


In [9]:

# Cellule 8 — baseline à figer avant entraînement

baseline_metrics = evaluator(model)
evaluator.set_baseline(baseline_metrics)

print(baseline_metrics)

baseline_score = baseline_metrics["robust_objective_score"]
baseline_retrieval_score = baseline_metrics["robust_retrieval_metric"]
baseline_sts_score = baseline_metrics["robust_sts_metric"]
baseline_retention_score = baseline_metrics["robust_retention_score"]

print({
    "baseline_objective_score": round(baseline_score, 6),
    "baseline_retrieval_ndcg@10": round(baseline_retrieval_score, 6),
    "baseline_sts": round(baseline_sts_score, 6),
    "baseline_retention": round(baseline_retention_score, 6),
    "selection_objective": SAFE_OBJECTIVE,
})


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:18<00:00, 18.43s/it]


Baseline robuste figée: {'retrieval': 0.8369931423720339, 'sts': 0.8121484291996951, 'retention': 1.0, 'objective': 0.8796840213673391}
{'miracl-fr-heldout_cosine_accuracy@1': 0.75, 'miracl-fr-heldout_cosine_accuracy@3': 0.8866666666666667, 'miracl-fr-heldout_cosine_accuracy@5': 0.9466666666666667, 'miracl-fr-heldout_cosine_accuracy@10': 0.9866666666666667, 'miracl-fr-heldout_cosine_precision@1': 0.75, 'miracl-fr-heldout_cosine_precision@3': 0.5011111111111112, 'miracl-fr-heldout_cosine_precision@5': 0.362, 'miracl-fr-heldout_cosine_precision@10': 0.2036666666666667, 'miracl-fr-heldout_cosine_recall@1': 0.4318597883597884, 'miracl-fr-heldout_cosine_recall@3': 0.7544867724867724, 'miracl-fr-heldout_cosine_recall@5': 0.8730264550264549, 'miracl-fr-heldout_cosine_recall@10': 0.9608386243386242, 'miracl-fr-heldout_cosine_ndcg@10': 0.8369931423720339, 'miracl-fr-heldout_cosine_mrr@10': 0.8321878306878306, 'miracl-fr-heldout_cosine_map@100': 0.7763075283005372, 'fr-sts-heldout_pearson_cosine

In [10]:
# Cellule 9 — test hard negative avant entraînement

row = train_ds[0]

q = "query: " + row["anchor"]
p = "passage: " + row["positive"]
n = "passage: " + row["negative"]

emb = model.encode([q, p, n], normalize_embeddings=True)

positive_score = float(emb[0] @ emb[1])
negative_score = float(emb[0] @ emb[2])

print({
    "positive_score": positive_score,
    "negative_score": negative_score,
    "margin": positive_score - negative_score,
})

assert positive_score > negative_score, "La baseline ne classe pas ce triplet correctement"

{'positive_score': 0.8695799112319946, 'negative_score': 0.8043878674507141, 'margin': 0.06519204378128052}


In [11]:

# Cellule 10 — création du vrai PeftModel AVANT le trainer et l'optimiseur
# Version SAFE: adapter plus petit + moins de modules cibles pour réduire le drift global.

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=4,
    lora_alpha=4,
    lora_dropout=0.15,
    # Conservative v3: seulement query/value des deux dernières couches.
    # Les couches basses/moyennes portent beaucoup de géométrie multilingue; on les laisse intactes.
    target_modules=(
        r".*encoder\.layer\.(10|11)\.attention\.self\.(query|value)"
    ),
    bias="none",
)

model[0].model = get_peft_model(model[0].auto_model, lora_config)
assert isinstance(model[0].auto_model, PeftModel)
print("Backbone PEFT:", type(model[0].auto_model).__name__)


Backbone PEFT: PeftModelForFeatureExtraction


In [12]:
# Cellule 11 — garde-fous AVANT entraînement

backbone = model[0].auto_model
lora_all = [(name, p) for name, p in backbone.named_parameters() if "lora_" in name.lower()]
lora_trainable = [(name, p) for name, p in lora_all if p.requires_grad]
non_lora_trainable = [
    name for name, p in backbone.named_parameters()
    if p.requires_grad and "lora_" not in name.lower()
]
lora_b_before = {
    name: p.detach().float().cpu().clone()
    for name, p in lora_all if "lora_b" in name.lower()
}

trainable = sum(p.numel() for _, p in lora_trainable)
total = sum(p.numel() for p in model.parameters())
initial_b_max = max(t.abs().max().item() for t in lora_b_before.values())

print("Tenseurs LoRA:", len(lora_all))
print("Paramètres LoRA entraînables:", f"{trainable:,}")
print("Part entraînable:", f"{100 * trainable / total:.3f}%")
print("max abs LoRA_B initial:", initial_b_max)

assert lora_all and lora_trainable, "LoRA absent ou gelé."
assert not non_lora_trainable, f"Paramètres non-LoRA entraînables: {non_lora_trainable[:5]}"
assert initial_b_max == 0.0, "LoRA_B devrait être nul à l'initialisation."

Tenseurs LoRA: 8
Paramètres LoRA entraînables: 24,576
Part entraînable: 0.009%
max abs LoRA_B initial: 0.0


In [13]:

# Cellule 11 — pertes adaptées + distillation anti-régression

class WeightedLoss(torch.nn.Module):
    """Multiplie une loss ST sans modifier son API."""

    def __init__(self, wrapped_loss, weight):
        super().__init__()
        self.wrapped_loss = wrapped_loss
        self.weight = float(weight)

    def forward(self, *args, **kwargs):
        value = self.wrapped_loss(*args, **kwargs)
        if isinstance(value, dict):
            return {key: tensor * self.weight for key, tensor in value.items()}
        return value * self.weight


miracl_loss = CachedMultipleNegativesRankingLoss(
    model=model,
    scale=10.0,
    mini_batch_size=8,
)
paraphrase_loss = CachedMultipleNegativesRankingLoss(
    model=model,
    scale=10.0,
    mini_batch_size=8,
)
replay_loss = CachedMultipleNegativesRankingLoss(
    model=model,
    scale=10.0,
    mini_batch_size=8,
)
margin_distill_loss = MarginMSELoss(model=model)
similarity_loss = CosineSimilarityLoss(model=model)
baseline_distill_loss = EmbedDistillLoss(model=model, distance_metric="cosine")

loss = {
    "miracl": WeightedLoss(miracl_loss, LOSS_WEIGHTS["miracl"]),
    "miracl_margin": WeightedLoss(margin_distill_loss, LOSS_WEIGHTS["miracl_margin"]),
    "fr_sts": WeightedLoss(similarity_loss, LOSS_WEIGHTS["fr_sts"]),
    "fr_paraphrase": WeightedLoss(paraphrase_loss, LOSS_WEIGHTS["fr_paraphrase"]),
    "fr_nli": WeightedLoss(similarity_loss, LOSS_WEIGHTS["fr_nli"]),
    "multilingual_replay": WeightedLoss(replay_loss, LOSS_WEIGHTS["multilingual_replay"]),
    "baseline_distill": WeightedLoss(baseline_distill_loss, LOSS_WEIGHTS["baseline_distill"]),
}

assert set(loss) == set(train_datasets)
print({name: {"loss": type(dataset_loss.wrapped_loss).__name__, "weight": dataset_loss.weight} for name, dataset_loss in loss.items()})


{'miracl': {'loss': 'CachedMultipleNegativesRankingLoss', 'weight': 0.08}, 'miracl_margin': {'loss': 'MarginMSELoss', 'weight': 0.35}, 'fr_sts': {'loss': 'CosineSimilarityLoss', 'weight': 0.02}, 'fr_paraphrase': {'loss': 'CachedMultipleNegativesRankingLoss', 'weight': 0.02}, 'fr_nli': {'loss': 'CosineSimilarityLoss', 'weight': 0.005}, 'multilingual_replay': {'loss': 'CachedMultipleNegativesRankingLoss', 'weight': 0.12}, 'baseline_distill': {'loss': 'EmbedDistillLoss', 'weight': 8.0}}


In [14]:

# Cellule 12 — arguments d'entraînement SAFE

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,

    fp16=(device == "cuda"),
    bf16=False,

    batch_sampler=BatchSamplers.NO_DUPLICATES,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    # Ne pas utiliser les checkpoints ST automatiques pour ce PEFT imbriqué.
    # Avec les versions installées, leur rechargement peut remettre LoRA_B à zéro.
    save_strategy="no",

    logging_steps=25,

    load_best_model_at_end=False,

    report_to="none",

    # Prompts E5 par dataset: retrieval asymétrique, autres tâches symétriques.
    # baseline_distill contient déjà le préfixe dans la colonne text.
    prompts={
        "miracl": {
            "anchor": "query: ",
            "positive": "passage: ",
            "negative": "passage: ",
        },
        "miracl_margin": {
            "anchor": "query: ",
            "positive": "passage: ",
            "negative": "passage: ",
        },
        "fr_sts": {"sentence1": "query: ", "sentence2": "query: "},
        "fr_paraphrase": {"anchor": "query: ", "positive": "query: "},
        "fr_nli": {"sentence1": "query: ", "sentence2": "query: "},
        "multilingual_replay": {"anchor": "query: ", "positive": "query: "},
        "baseline_distill": {"text": ""},
    },

    dataloader_num_workers=0,
    seed=SEED,
)

print("Arguments SAFE OK")


The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Arguments SAFE OK


In [15]:

class LoraTrainingGuardCallback(TrainerCallback):
    """Vérifie LoRA_B et ne conserve qu'un adapter qui passe les garde-fous."""

    def __init__(self, best_adapter_dir, metric_name):
        self.gradient_checked = False
        self.weight_checked = False
        self.best_adapter_dir = best_adapter_dir
        self.metric_name = metric_name
        self.best_score = float("-inf")
        self.best_step = None
        self.best_lora_state = None
        self.best_metrics = None
        self.bad_eval_count = 0
        self.stop_reason = None

    @staticmethod
    def metric(metrics, key, default=None):
        if metrics is None:
            return default
        if key in metrics:
            return metrics[key]
        if key.startswith("eval_") and key[5:] in metrics:
            return metrics[key[5:]]
        prefixed = "eval_" + key
        if prefixed in metrics:
            return metrics[prefixed]
        return default

    @staticmethod
    def max_abs_lora_b(model):
        values = [
            p.detach().float().abs().max().item()
            for name, p in model[0].auto_model.named_parameters()
            if "lora_b" in name.lower()
        ]
        return max(values, default=0.0)

    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        # Ce hook voit les gradients avant optimizer.step() et model.zero_grad().
        if not self.gradient_checked:
            lora_b_gradients = [
                p.grad.detach().float()
                for name, p in model[0].auto_model.named_parameters()
                if "lora_b" in name.lower() and p.grad is not None
            ]
            grad_max = max((g.abs().max().item() for g in lora_b_gradients), default=0.0)
            print(f"[LoRA gradient check] max abs grad LoRA_B = {grad_max:.3e}")
            if not np.isfinite(grad_max) or grad_max <= 0.0:
                raise RuntimeError(
                    f"Gradients LoRA_B invalides avant optimizer.step(): {grad_max}"
                )
            self.gradient_checked = True
        return control

    def on_optimizer_step(self, args, state, control, model=None, optimizer=None, **kwargs):
        # Avec warmup, le LR du premier step peut valoir 0: attendre le premier LR positif.
        learning_rate = max(group["lr"] for group in optimizer.param_groups)
        if not self.weight_checked and learning_rate > 0.0:
            lora_b = [
                p.detach().float()
                for name, p in model[0].auto_model.named_parameters()
                if "lora_b" in name.lower()
            ]
            max_abs = max(p.abs().max().item() for p in lora_b)
            print(
                f"[LoRA weight check step {state.global_step + 1}] "
                f"lr={learning_rate:.3e}, max abs LoRA_B={max_abs:.3e}"
            )
            if not np.isfinite(max_abs) or max_abs <= 0.0:
                raise RuntimeError(
                    f"LoRA_B invalide malgré un optimizer step à LR positif: {max_abs}"
                )
            self.weight_checked = True
        return control

    def consider(self, model, metrics, step):
        score = self.metric(metrics, self.metric_name, float("-inf"))
        passes = float(self.metric(metrics, "robust_passes_guardrails", 0.0))
        delta = float(self.metric(metrics, "robust_objective_delta", float("-inf")))
        penalty = float(self.metric(metrics, "robust_penalty", 0.0))

        if not np.isfinite(score):
            print(f"[reject adapter] step={step}: score invalide")
            return False
        if passes < 1.0:
            print(
                f"[reject adapter] step={step}: guardrails fail, "
                f"delta={delta:.6f}, penalty={penalty:.6f}, score={score:.6f}"
            )
            return False
        if delta < MIN_ROBUST_DELTA:
            print(
                f"[reject adapter] step={step}: delta objectif {delta:.6f} "
                f"< MIN_ROBUST_DELTA {MIN_ROBUST_DELTA:.6f}"
            )
            return False
        if score <= self.best_score:
            return False

        backbone = model[0].auto_model
        self.best_score = float(score)
        self.best_step = step
        self.best_metrics = dict(metrics)
        self.best_lora_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in backbone.state_dict().items()
            if "lora_" in name.lower()
        }
        backbone.save_pretrained(self.best_adapter_dir, safe_serialization=True)
        print(
            f"[best safe adapter] step={step} robust_score={score:.6f} "
            f"objective_delta={delta:.6f}"
        )
        return True

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is not None:
            self.consider(model, metrics, state.global_step)
            retention = float(self.metric(metrics, "robust_retention_cosine", 1.0))
            pairwise = float(self.metric(metrics, "robust_pairwise_retention", 1.0))
            lora_b_max = self.max_abs_lora_b(model)
            hard_failure = (
                retention < HARD_STOP_RETENTION_COSINE
                or pairwise < HARD_STOP_PAIRWISE_RETENTION
                or lora_b_max > MAX_LORA_B_ABS
            )
            if hard_failure:
                self.bad_eval_count += 1
                print(
                    f"[early-stop watch] bad_eval={self.bad_eval_count}/{MAX_BAD_EVALS} "
                    f"retention={retention:.4f}, pairwise={pairwise:.4f}, "
                    f"max_lora_b={lora_b_max:.4e}"
                )
            else:
                self.bad_eval_count = 0
            if self.bad_eval_count >= MAX_BAD_EVALS:
                self.stop_reason = (
                    f"Drift excessif: retention={retention:.4f}, "
                    f"pairwise={pairwise:.4f}, max_lora_b={lora_b_max:.4e}"
                )
                print(f"[early stop] {self.stop_reason}")
                control.should_training_stop = True
        return control

    def restore_best(self, model):
        assert self.best_lora_state is not None, "Aucun adapter n'a passé les garde-fous."
        model[0].auto_model.load_state_dict(self.best_lora_state, strict=False)
        print(
            f"Adapter SAFE restauré: step={self.best_step}, "
            f"robust_score={self.best_score:.6f}"
        )


BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best-adapter-heldout")
HELDOUT_METRIC = "robust_score"
lora_training_guard = LoraTrainingGuardCallback(BEST_ADAPTER_DIR, HELDOUT_METRIC)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_datasets,
    loss=loss,
    evaluator=evaluator,
    callbacks=[lora_training_guard],
)

# Construction explicite de l'optimiseur, puis preuve que tous les LoRA trainables y sont.
trainer.create_optimizer()
optimizer_param_ids = {
    id(p) for group in trainer.optimizer.param_groups for p in group["params"]
}
missing_from_optimizer = [name for name, p in lora_trainable if id(p) not in optimizer_param_ids]
assert not missing_from_optimizer, f"LoRA absent de l'optimiseur: {missing_from_optimizer[:5]}"
print("Optimiseur OK: tous les paramètres LoRA entraînables sont présents.")

trainer.train()

lora_b_after = {
    name: p.detach().float().cpu()
    for name, p in model[0].auto_model.named_parameters()
    if "lora_b" in name.lower()
}
max_b_after = max(t.abs().max().item() for t in lora_b_after.values())
max_b_delta = max(
    (tensor - lora_b_before[name]).abs().max().item()
    for name, tensor in lora_b_after.items()
)

assert lora_training_guard.gradient_checked
assert lora_training_guard.weight_checked
assert np.isfinite(max_b_after) and np.isfinite(max_b_delta)
assert max_b_after > 0.0 and max_b_delta > 0.0
print({"max_abs_LoRA_B": max_b_after, "max_delta_LoRA_B": max_b_delta})
print("Training terminé: les poids LoRA ont réellement changé.")


Computing widget examples:   0%|          | 0/4 [00:00<?, ?example/s]

Optimiseur OK: tous les paramètres LoRA entraînables sont présents.
[LoRA gradient check] max abs grad LoRA_B = 1.500e-03


Step,Training Loss,Validation Loss,Miracl-fr-heldout Cosine Accuracy@1,Miracl-fr-heldout Cosine Accuracy@3,Miracl-fr-heldout Cosine Accuracy@5,Miracl-fr-heldout Cosine Accuracy@10,Miracl-fr-heldout Cosine Precision@1,Miracl-fr-heldout Cosine Precision@3,Miracl-fr-heldout Cosine Precision@5,Miracl-fr-heldout Cosine Precision@10,Miracl-fr-heldout Cosine Recall@1,Miracl-fr-heldout Cosine Recall@3,Miracl-fr-heldout Cosine Recall@5,Miracl-fr-heldout Cosine Recall@10,Miracl-fr-heldout Cosine Ndcg@10,Miracl-fr-heldout Cosine Mrr@10,Miracl-fr-heldout Cosine Map@100,Fr-sts-heldout Pearson Cosine,Fr-sts-heldout Spearman Cosine,Robust Score,Robust Objective Score,Robust Objective Delta,Robust Penalty,Robust Passes Guardrails,Robust Retrieval Metric,Robust Sts Metric,Robust Retrieval Delta,Robust Sts Delta,Robust Retention Cosine,Robust Pairwise Retention,Robust Retention Score,Sequential Score
100,0.115394,No log,0.753333,0.886667,0.946667,0.986667,0.753333,0.501111,0.361333,0.203667,0.435193,0.754487,0.872193,0.960839,0.837663,0.833299,0.777238,0.816993,0.812122,0.000261,0.879945,0.000261,0.000000,1.000000,0.837663,0.812122,0.000670,-0.000027,0.999994,0.999886,0.999886,0.000261
200,0.129407,No log,0.753333,0.886667,0.950000,0.986667,0.753333,0.502222,0.362667,0.204000,0.435193,0.755598,0.876360,0.961950,0.838934,0.833966,0.778490,0.817022,0.812137,0.000741,0.880425,0.000741,0.000000,1.000000,0.838934,0.812137,0.001941,-0.000012,0.999993,0.999569,0.999569,0.000741
300,0.116757,No log,0.753333,0.886667,0.950000,0.986667,0.753333,0.502222,0.362000,0.204000,0.435193,0.755598,0.875526,0.961950,0.838917,0.833966,0.778386,0.817084,0.812181,0.000624,0.880308,0.000624,0.000000,1.000000,0.838917,0.812181,0.001924,0.000033,0.999992,0.999168,0.999168,0.000624
400,0.136377,No log,0.750000,0.886667,0.950000,0.986667,0.750000,0.501111,0.362000,0.203667,0.431860,0.754487,0.875526,0.960839,0.837069,0.832299,0.776257,0.817142,0.812261,-0.005352,0.879379,-0.000305,0.005048,0.000000,0.837069,0.812261,0.000075,0.000112,0.999989,0.998777,0.998777,-0.005352
500,0.126586,No log,0.750000,0.890000,0.950000,0.986667,0.750000,0.502222,0.361333,0.203667,0.431860,0.757820,0.874693,0.960839,0.836851,0.832021,0.776013,0.817166,0.812247,-0.007869,0.879151,-0.000534,0.007335,0.000000,0.836851,0.812247,-0.000142,0.000099,0.999986,0.998353,0.998353,-0.007869
600,0.111790,No log,0.750000,0.890000,0.950000,0.986667,0.750000,0.504444,0.361333,0.204000,0.431860,0.759765,0.874693,0.961950,0.837082,0.831688,0.775835,0.817228,0.812312,-0.007740,0.879162,-0.000522,0.007218,0.000000,0.837082,0.812312,0.000089,0.000163,0.999982,0.997992,0.997992,-0.007740
700,0.135762,No log,0.753333,0.890000,0.950000,0.986667,0.753333,0.503333,0.362000,0.204000,0.435193,0.758931,0.875526,0.961950,0.839616,0.834799,0.779299,0.817300,0.812393,0.000536,0.880220,0.000536,0.000000,1.000000,0.839616,0.812393,0.002623,0.000245,0.999977,0.997648,0.997648,0.000536
800,0.132286,No log,0.753333,0.886667,0.950000,0.986667,0.753333,0.501111,0.362000,0.204000,0.435193,0.754487,0.875526,0.961950,0.838411,0.833243,0.777846,0.817311,0.812410,-0.003230,0.879572,-0.000112,0.003118,0.000000,0.838411,0.812410,0.001417,0.000261,0.999972,0.997283,0.997283,-0.003230
900,0.111615,No log,0.753333,0.886667,0.950000,0.986667,0.753333,0.500000,0.362667,0.204000,0.435193,0.752820,0.876360,0.961950,0.838488,0.833799,0.777746,0.817316,0.812446,-0.003637,0.879535,-0.000149,0.003488,0.000000,0.838488,0.812446,0.001495,0.000297,0.999967,0.997013,0.997013,-0.003637
1000,0.117304,No log,0.753333,0.886667,0.950000,0.986667,0.753333,0.501111,0.362667,0.204000,0.435193,0.753653,0.876360,0.961950,0.839236,0.834521,0.778854,0.817334,0.812461,-0.000709,0.879801,0.000117,0.000826,0.000000,0.839236,0.812461,0.002243,0.000313,0.999963,0.996766,0.996766,-0.000709


[LoRA weight check step 2] lr=1.274e-08, max abs LoRA_B=1.276e-08


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:07<00:00,  7.47s/it]


[reject adapter] step=1562: guardrails fail, delta=-0.000074, penalty=0.002740, score=-0.002814
{'max_abs_LoRA_B': 0.0017372006550431252, 'max_delta_LoRA_B': 0.0017372006550431252}
Training terminé: les poids LoRA ont réellement changé.


In [16]:

# Cellule 15 — sélection et verdict sur le holdout sans fuite / sans régression proxy

last_metrics = evaluator(model)
last_score = last_metrics["robust_score"]
lora_training_guard.consider(model, last_metrics, trainer.state.global_step)

if lora_training_guard.best_lora_state is None:
    print({
        "last_robust_score": round(last_score, 6),
        "last_objective_delta": round(last_metrics["robust_objective_delta"], 6),
        "last_retrieval_delta": round(last_metrics["robust_retrieval_delta"], 6),
        "last_sts_delta": round(last_metrics["robust_sts_delta"], 6),
        "last_retention_cosine": round(last_metrics["robust_retention_cosine"], 6),
        "last_pairwise_retention": round(last_metrics["robust_pairwise_retention"], 6),
        "early_stop_reason": lora_training_guard.stop_reason,
    })
    raise RuntimeError(
        "Aucun checkpoint n'a passé les garde-fous anti-régression. "
        "Ne pas exporter et ne pas lancer MTEB complet; diminuer encore LR/LoRA ou augmenter replay/distillation."
    )

lora_training_guard.restore_best(model)

# Recalcul après restauration: le modèle ensuite sauvegardé est bien le meilleur SAFE.
selected_metrics = evaluator(model)
final_score = selected_metrics["robust_score"]
final_delta = selected_metrics["robust_objective_delta"]
assert selected_metrics["robust_passes_guardrails"] == 1.0
assert final_delta >= MIN_ROBUST_DELTA
assert abs(final_score - lora_training_guard.best_score) < 1e-6

print({
    "selection_objective": SAFE_OBJECTIVE,
    "baseline_objective_score": round(baseline_score, 6),
    "last_robust_score": round(last_score, 6),
    "selected_robust_score": round(final_score, 6),
    "selected_objective_delta": round(final_delta, 6),
    "selected_retrieval_ndcg@10": round(selected_metrics["robust_retrieval_metric"], 6),
    "selected_retrieval_delta": round(selected_metrics["robust_retrieval_delta"], 6),
    "selected_sts": round(selected_metrics["robust_sts_metric"], 6),
    "selected_sts_delta": round(selected_metrics["robust_sts_delta"], 6),
    "selected_retention_cosine": round(selected_metrics["robust_retention_cosine"], 6),
    "selected_pairwise_retention": round(selected_metrics["robust_pairwise_retention"], 6),
    "selected_step": lora_training_guard.best_step,
})

print("WIN SAFE held-out: candidat valable pour une mini-éval puis MTEB français")


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:07<00:00,  7.62s/it]


[reject adapter] step=1562: guardrails fail, delta=-0.000074, penalty=0.002740, score=-0.002814
Adapter SAFE restauré: step=200, robust_score=0.000741


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:07<00:00,  7.40s/it]


{'selection_objective': 'global_safe', 'baseline_objective_score': 0.879684, 'last_robust_score': -0.002814, 'selected_robust_score': 0.000741, 'selected_objective_delta': 0.000741, 'selected_retrieval_ndcg@10': 0.838934, 'selected_retrieval_delta': 0.001941, 'selected_sts': 0.812137, 'selected_sts_delta': -1.2e-05, 'selected_retention_cosine': 0.999993, 'selected_pairwise_retention': 0.999569, 'selected_step': 200}
WIN SAFE held-out: candidat valable pour une mini-éval puis MTEB français


In [17]:
# Cellule 16 — test qualitatif final

tests = [
    {
        "query": "Qui était la princesse Diana ?",
        "positive": "Diana Spencer était une membre de la famille royale britannique.",
        "negative": "La photosynthèse permet aux plantes de produire de l'énergie.",
    },
    {
        "query": "Quelle est la capitale de la France ?",
        "positive": "Paris est la capitale de la France.",
        "negative": "Le Japon est un archipel situé en Asie de l'Est.",
    },
]

for item in tests:
    vectors = model.encode(
        [
            "query: " + item["query"],
            "passage: " + item["positive"],
            "passage: " + item["negative"],
        ],
        normalize_embeddings=True,
    )

    pos = float(vectors[0] @ vectors[1])
    neg = float(vectors[0] @ vectors[2])

    print({
        "query": item["query"],
        "positive": round(pos, 4),
        "negative": round(neg, 4),
        "margin": round(pos - neg, 4),
    })

    assert pos > neg

{'query': 'Qui était la princesse Diana ?', 'positive': 0.8468, 'negative': 0.7085, 'margin': 0.1383}
{'query': 'Quelle est la capitale de la France ?', 'positive': 0.8836, 'negative': 0.7402, 'margin': 0.1434}


In [18]:

import shutil

ADAPTER_DIR = "./e5-fr-conservative-lora-adapter"
FINAL_DIR = "./e5-fr-conservative-lora-merged"
probe_texts = [
    "query: Quelle est la capitale de la France ?",
    "passage: Paris est la capitale de la France.",
]


In [19]:
# Cellule 17 — export robuste depuis le meilleur adapter sauvegardé
# On ne dépend pas du type de wrapper laissé en mémoire par Trainer/Accelerate.



assert os.path.isfile(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json"))
assert os.path.isfile(os.path.join(BEST_ADAPTER_DIR, "adapter_model.safetensors"))

# Libère les états optimizer/gradients GPU devenus inutiles.
model.to("cpu")
del trainer, loss, miracl_loss, paraphrase_loss, replay_loss, margin_distill_loss, similarity_loss, baseline_distill_loss, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Recharge explicitement le meilleur adapter sur un backbone E5 propre, en CPU.
export_model = SentenceTransformer(MODEL_NAME, device="cpu")
export_model.max_seq_length = MAX_SEQ_LENGTH
export_model[0].model = PeftModel.from_pretrained(
    export_model[0].auto_model,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)
peft_backbone = export_model[0].auto_model
lora_b_export = [
    parameter.detach().float()
    for name, parameter in peft_backbone.named_parameters()
    if "lora_b" in name.lower()
]
max_lora_b_export = max((p.abs().max().item() for p in lora_b_export), default=0.0)
assert np.isfinite(max_lora_b_export) and max_lora_b_export > 0.0

reference_embeddings = export_model.encode(probe_texts, normalize_embeddings=True)

# Supprime d'éventuels fichiers résiduels d'anciens exports cassés.
for output_path in (ADAPTER_DIR, FINAL_DIR):
    if os.path.isdir(output_path):
        shutil.rmtree(output_path)

peft_backbone.save_pretrained(ADAPTER_DIR, safe_serialization=True)
export_model[0].model = peft_backbone.merge_and_unload(safe_merge=True)
export_model.save_pretrained(FINAL_DIR)

reloaded_model = SentenceTransformer(FINAL_DIR, device="cpu")
reloaded_embeddings = reloaded_model.encode(probe_texts, normalize_embeddings=True)
reload_max_diff = float(np.max(np.abs(reference_embeddings - reloaded_embeddings)))
assert np.allclose(reference_embeddings, reloaded_embeddings, rtol=1e-4, atol=1e-5)

# Le modèle global redevient le modèle fusionné propre, en CPU.
model = export_model
del reloaded_model
gc.collect()

print({
    "best_step": lora_training_guard.best_step,
    "best_combined_score": lora_training_guard.best_score,
    "max_abs_LoRA_B_export": max_lora_b_export,
    "reload_max_diff": reload_max_diff,
})
print("Adapter PEFT:", ADAPTER_DIR)
print("Modèle fusionné MTEB:", FINAL_DIR)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'best_step': 200, 'best_combined_score': 0.0007410968658640993, 'max_abs_LoRA_B_export': 0.0002411393215879798, 'reload_max_diff': 6.332993507385254e-08}
Adapter PEFT: ./e5-fr-conservative-lora-adapter
Modèle fusionné MTEB: ./e5-fr-conservative-lora-merged


## 10. Rechargement et vérification

In [ ]:
import ctypes
import gc
import os
import torch
from sentence_transformers import SentenceTransformer

# Le training/export laisse plusieurs alias vers des backbones E5 en mémoire.
# On les supprime avant MTEB pour ne conserver qu'un seul modèle sur le GPU.
for _name in (
    "model", "base_model", "st_model", "export_model",
    "reloaded_model", "adapter_model", "benchmark", "task", "results",
    "peft_backbone", "backbone", "lora_b", "lora_b_export",
    "baseline_model", "finetuned_model", "adapter_embeddings",
    "merged_embeddings", "reference_embeddings", "reloaded_embeddings",
    "baseline_probe", "finetuned_probe", "test_q", "test_d",
    "ds", "fr_sts_splits", "fr_nli_raw", "replay_raw",
    "train_ds", "eval_ds", "sts_source", "sts_train",
    "paraphrase_source", "paraphrase_train", "nli_source", "nli_parts",
    "nli_train", "replay_parts", "replay_train", "train_datasets",
    "all_training_texts", "heldout_texts", "eval_positive_texts",
    "retrieval_evaluator", "sts_validation", "sts_evaluator", "evaluator",
    "lora_b_before", "lora_b_after", "q_emb", "p_emb", "emb",
    "lora_training_guard",
):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if os.name == "nt":
    ctypes.windll.psapi.EmptyWorkingSet(ctypes.windll.kernel32.GetCurrentProcess())

# Valeurs autonomes: cette section peut être lancée après un restart kernel,
# sans réexécuter le téléchargement des datasets ni le training.
MODEL_NAME = globals().get("MODEL_NAME", "intfloat/multilingual-e5-base")
ADAPTER_DIR = globals().get("ADAPTER_DIR", "./e5-fr-miracl-lora-adapter")
FINAL_DIR = globals().get("FINAL_DIR", "./e5-fr-miracl-lora-merged")
MAX_SEQ_LENGTH = globals().get("MAX_SEQ_LENGTH", 512)
device = "cuda" if torch.cuda.is_available() else "cpu"

BASELINE_DIR_OR_NAME = MODEL_NAME
FINETUNED_DIR = FINAL_DIR

# MTEB n'a besoin que du modèle fusionné. Le baseline et l'adapter ne sont pas
# rechargés ici: cela évite 1 à 2 Go de RAM temporaire que Python rend mal à Windows.
# FP16 économise ~0.5 Go de VRAM mais modifie sensiblement certains clusterings.
USE_FP16_EVAL = False
EVAL_TORCH_DTYPE = (
    torch.float16 if USE_FP16_EVAL and torch.cuda.is_available() else torch.float32
)
LOAD_MODEL_IN_NOTEBOOK = False
if LOAD_MODEL_IN_NOTEBOOK:
    finetuned_model = SentenceTransformer(
        FINETUNED_DIR,
        device=device,
        model_kwargs={"torch_dtype": EVAL_TORCH_DTYPE},
    )
else:
    finetuned_model = None
    print("Mode SAFE: aucun modèle conservé dans le kernel Jupyter.")


In [ ]:
if finetuned_model is not None:
    merged_lora_params = [
        name for name, _ in finetuned_model[0].auto_model.named_parameters()
        if "lora_" in name.lower()
    ]
    assert not merged_lora_params
    print("Modèle fusionné sans couche LoRA résiduelle.")
else:
    print("Vérification locale ignorée en mode SAFE.")

In [ ]:
# Diagnostic lourd facultatif. Laisser False pendant MTEB.
RUN_RELOAD_DIAGNOSTICS = False
if RUN_RELOAD_DIAGNOSTICS:
    from peft import PeftModel
    adapter_model = SentenceTransformer(MODEL_NAME, device="cpu")
    adapter_model.max_seq_length = MAX_SEQ_LENGTH
    adapter_model[0].model = PeftModel.from_pretrained(
        adapter_model[0].auto_model, ADAPTER_DIR, is_trainable=False
    )
    adapter_model.eval()
else:
    print("Diagnostic adapter ignoré pour économiser la RAM.")

In [ ]:
if RUN_RELOAD_DIAGNOSTICS:
    backbone = adapter_model[0].auto_model
    lora_b = [
        (name, p.detach().float())
        for name, p in backbone.named_parameters()
        if "lora_b" in name.lower()
    ]
    max_lora_b = max(p.abs().max().item() for _, p in lora_b)
    assert lora_b and max_lora_b > 0.0
    adapter_embeddings = adapter_model.encode(probe_texts, normalize_embeddings=True)
    merged_embeddings = finetuned_model.encode(probe_texts, normalize_embeddings=True)
    assert np.allclose(adapter_embeddings, merged_embeddings, rtol=1e-3, atol=2e-4)
    print("Adapter PEFT et modèle FP16 fusionné: embeddings cohérents.")
    del adapter_model, backbone, lora_b, adapter_embeddings, merged_embeddings
    gc.collect()
else:
    print("Validation adapter déjà effectuée pendant l'export.")

In [ ]:
if finetuned_model is not None:
    print("Fine-tuned dim:", finetuned_model.get_embedding_dimension())
    print("dtype évaluation:", next(finetuned_model.parameters()).dtype)
else:
    print("Le worker isolé chargera et libérera le modèle tâche par tâche.")

In [ ]:
# L'évaluation SAFE est autonome dans la cellule suivante.
# Aucun import MTEB/PyTorch lourd n'est nécessaire dans le kernel Jupyter.
print("Préparation du superviseur MTEB SAFE.")

In [ ]:

import hashlib


def short_file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b=""):
            digest.update(chunk)
    return digest.hexdigest()[:12]


MODEL_REVISION = "lora-" + short_file_sha256(
    os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
)
MODEL_LABEL = "local/multilingual-e5-base-fr-conservative-lora"
RUN_NAME = f"e5_fr_conservative_{MODEL_REVISION}"
print({"model_label": MODEL_LABEL, "revision": MODEL_REVISION, "run": RUN_NAME})


In [ ]:
# Pour mesurer la baseline dans un second run, utiliser à la place :
# MODEL_LABEL = MODEL_NAME
# MODEL_REVISION = "upstream-baseline"
# RUN_NAME = "e5_baseline_upstream"
# base_model = baseline_model

In [1]:
# Evaluation MTEB SAFE: une tache par processus jetable.
# Cette cellule est autonome apres un restart kernel et reprend les JSON existants.
import ctypes
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import psutil

MODEL_NAME = "intfloat/multilingual-e5-base"
MODEL_LABEL = "local/multilingual-e5-base-fr-conservative-lora"
BASE_DIR = Path.cwd()
if not (BASE_DIR / "mteb_safe_worker.py").is_file():
    candidate = Path(r"F:\codeAI\llm\emebedding")
    if (candidate / "mteb_safe_worker.py").is_file():
        BASE_DIR = candidate

FINETUNED_DIR = (BASE_DIR / "e5-fr-conservative-lora-merged").resolve()
ADAPTER_FILE = (BASE_DIR / "e5-fr-conservative-lora-adapter" / "adapter_model.safetensors").resolve()
WORKER_PATH = (BASE_DIR / "mteb_safe_worker.py").resolve()
PYTHON_EXECUTABLE = BASE_DIR.parent / ".venv" / "Scripts" / "python.exe"
if not PYTHON_EXECUTABLE.is_file():
    PYTHON_EXECUTABLE = Path(sys.executable)

assert FINETUNED_DIR.is_dir(), FINETUNED_DIR
assert ADAPTER_FILE.is_file(), ADAPTER_FILE
assert WORKER_PATH.is_file(), WORKER_PATH

def short_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()[:12]

MODEL_REVISION = "lora-" + short_sha256(ADAPTER_FILE)
GPU_EVAL_MODE = "fp16"  # "fp16" recommandé, "int8" expérimental bitsandbytes
RUN_NAME = f"e5_fr_conservative_{MODEL_REVISION}_gpu_{GPU_EVAL_MODE}"
BENCHMARK_LABEL = f"MTEB(fra)-safe v2 GPU-{GPU_EVAL_MODE} conservative-lora"
OUTPUT_FOLDER = BASE_DIR / "mteb_fra_results" / RUN_NAME
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER = OUTPUT_FOLDER / "logs"
LOG_FOLDER.mkdir(exist_ok=True)

# Les trois taches legacy responsables des crashes chargent de tres gros datasets.
# Leurs versions v2 officielles MTEB utilisent un echantillon borne a 2 048 textes.
SAFE_TASKS = [
    "AmazonReviewsClassification",
    "MasakhaNEWSClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "MTOPIntentClassification",
    "AlloProfClusteringP2P",
    "AlloProfClusteringS2S",
    "HALClusteringS2S.v2",
    "MasakhaNEWSClusteringP2P",
    "MasakhaNEWSClusteringS2S",
    "MLSUMClusteringP2P.v2",
    "MLSUMClusteringS2S.v2",
    "PawsXPairClassification",
    "AlloprofReranking",
    "SyntecReranking",
    "AlloprofRetrieval",
    "BSARDRetrieval",
    "MintakaRetrieval",
    "SyntecRetrieval",
    "XPQARetrieval",
    "SICKFr",
    "STSBenchmarkMultilingualSTS",
    "SummEvalFr",
    "STS22",
]

# Reglages GPU sobres pour RTX 3070 / 8 Go VRAM / 16 Go RAM.
# Si le PC hard-reset encore, ce n'est pas rattrapable par Python: repasser CPU_ONLY=True.
CPU_ONLY = False
CPU_THREADS = 4
GPU_BATCH_SIZE = 8
GPU_CHUNK_SIZE = 128
CPU_BATCH_SIZE = 16
CPU_CHUNK_SIZE = 256
GPU_COOLDOWN_SECONDS = 0.02
CPU_COOLDOWN_SECONDS = 0.0
MIN_FREE_RAM_GB = 1.25
MAX_WORKER_RSS_GB = 5.0
MAX_GPU_TEMP_C = 72
MAX_GPU_VRAM_GB = 6.4
MAX_GPU_POWER_W = 205.0
GPU_MEMORY_FRACTION = 0.82
GPU_ATTENTION_IMPLEMENTATION = "sdpa"  # "default" si SDPA pose probleme
EMPTY_CACHE_EVERY_CHUNK = True
ALLOW_PRECISION_FALLBACK = True
GPU_TASK_TIMEOUT_MIN = 180
CPU_TASK_TIMEOUT_MIN = 360
POLL_SECONDS = 3
RETRY_ON_CPU = True


def result_path(task_name):
    return OUTPUT_FOLDER / f"{task_name}.json"


def error_path(task_name):
    return OUTPUT_FOLDER / f"{task_name}_ERROR.json"


def valid_result(task_name):
    path = result_path(task_name)
    if not path.is_file():
        return False
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        return (
            data.get("run_name") == RUN_NAME
            and data.get("model_label") == MODEL_LABEL
            and data.get("model_revision") == MODEL_REVISION
            and isinstance(data.get("score"), (int, float))
        )
    except (OSError, json.JSONDecodeError):
        return False


def process_tree_rss_gb(pid):
    try:
        root = psutil.Process(pid)
        processes = [root] + root.children(recursive=True)
        return sum(p.memory_info().rss for p in processes if p.is_running()) / 1024**3
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0.0


def stop_process_tree(pid):
    try:
        root = psutil.Process(pid)
        processes = root.children(recursive=True) + [root]
    except psutil.NoSuchProcess:
        return
    for process in processes:
        try:
            process.terminate()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    _, alive = psutil.wait_procs(processes, timeout=5)
    for process in alive:
        try:
            process.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass


def gpu_snapshot():
    try:
        output = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=temperature.gpu,memory.used,power.draw",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            timeout=3,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
        parts = [part.strip() for part in output.strip().splitlines()[0].split(",")]
        return {
            "temperature_c": int(float(parts[0])),
            "memory_used_gb": float(parts[1]) / 1024.0,
            "power_w": float(parts[2]),
        }
    except Exception:
        return None


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temporary, path)


def run_worker(task_name, device):
    is_gpu = device == "cuda"
    batch_size = GPU_BATCH_SIZE if is_gpu else CPU_BATCH_SIZE
    chunk_size = GPU_CHUNK_SIZE if is_gpu else CPU_CHUNK_SIZE
    cooldown_seconds = GPU_COOLDOWN_SECONDS if is_gpu else CPU_COOLDOWN_SECONDS
    precision = GPU_EVAL_MODE if is_gpu else "fp32"
    log_path = LOG_FOLDER / f"{task_name}_{device}_{precision}.log"
    command = [
        str(PYTHON_EXECUTABLE),
        str(WORKER_PATH),
        "--task", task_name,
        "--model-dir", str(FINETUNED_DIR),
        "--model-label", MODEL_LABEL,
        "--model-revision", MODEL_REVISION,
        "--adapted-from", MODEL_NAME,
        "--result-path", str(result_path(task_name)),
        "--run-name", RUN_NAME,
        "--benchmark-label", BENCHMARK_LABEL,
        "--device", device,
        "--batch-size", str(batch_size),
        "--chunk-size", str(chunk_size),
        "--cooldown-seconds", str(cooldown_seconds),
        "--min-free-ram-gb", str(MIN_FREE_RAM_GB),
        "--cpu-threads", str(CPU_THREADS),
        "--precision", precision,
        "--gpu-memory-fraction", str(GPU_MEMORY_FRACTION),
        "--attention-implementation", GPU_ATTENTION_IMPLEMENTATION,
    ]
    if is_gpu and ALLOW_PRECISION_FALLBACK:
        command.append("--allow-precision-fallback")
    if is_gpu and EMPTY_CACHE_EVERY_CHUNK:
        command.append("--empty-cache-every-chunk")
    environment = os.environ.copy()
    environment.update({
        "OMP_NUM_THREADS": str(CPU_THREADS),
        "MKL_NUM_THREADS": str(CPU_THREADS),
        "OPENBLAS_NUM_THREADS": str(CPU_THREADS),
        "TOKENIZERS_PARALLELISM": "false",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:64",
        "CUDA_MODULE_LOADING": "LAZY",
        "NVIDIA_TF32_OVERRIDE": "1",
    })
    if device == "cpu":
        environment["CUDA_VISIBLE_DEVICES"] = ""
    else:
        environment["CUDA_VISIBLE_DEVICES"] = "0"
    creationflags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    timeout_seconds = 60 * (
        GPU_TASK_TIMEOUT_MIN if device == "cuda" else CPU_TASK_TIMEOUT_MIN
    )

    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(BASE_DIR),
            env=environment,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            creationflags=creationflags,
        )
        started = time.monotonic()
        last_report = started
        reason = None

        try:
            while process.poll() is None:
                time.sleep(POLL_SECONDS)
                elapsed = time.monotonic() - started
                free_ram_gb = psutil.virtual_memory().available / 1024**3
                worker_rss_gb = process_tree_rss_gb(process.pid)
                gpu = gpu_snapshot() if device == "cuda" else None
    
                if free_ram_gb < MIN_FREE_RAM_GB:
                    reason = f"RAM systeme basse ({free_ram_gb:.2f} Go)"
                elif worker_rss_gb > MAX_WORKER_RSS_GB:
                    reason = f"worker trop volumineux ({worker_rss_gb:.2f} Go)"
                elif gpu is not None and gpu["temperature_c"] >= MAX_GPU_TEMP_C:
                    reason = f"GPU trop chaud ({gpu['temperature_c']} C)"
                elif gpu is not None and gpu["memory_used_gb"] >= MAX_GPU_VRAM_GB:
                    reason = f"VRAM trop haute ({gpu['memory_used_gb']:.2f} Go)"
                elif gpu is not None and gpu["power_w"] >= MAX_GPU_POWER_W:
                    reason = f"puissance GPU trop haute ({gpu['power_w']:.0f} W)"
                elif elapsed > timeout_seconds:
                    reason = f"timeout ({elapsed / 60:.1f} min)"
    
                if reason:
                    stop_process_tree(process.pid)
                    break
    
                if time.monotonic() - last_report >= 30:
                    temp_text = (
                        f", GPU {gpu['temperature_c']} C, "
                        f"VRAM {gpu['memory_used_gb']:.1f} Go, "
                        f"{gpu['power_w']:.0f} W"
                        if gpu is not None else ""
                    )
                    print(
                        f"    {elapsed / 60:.1f} min, worker {worker_rss_gb:.1f} Go, "
                        f"RAM libre {free_ram_gb:.1f} Go{temp_text}",
                        flush=True,
                    )
                    last_report = time.monotonic()
    
        except KeyboardInterrupt:
            stop_process_tree(process.pid)
            raise

        return_code = process.poll()

    if reason:
        return False, reason, log_path
    if return_code != 0:
        return False, f"worker exit={return_code}", log_path
    if not valid_result(task_name):
        return False, "resultat absent ou invalide", log_path
    return True, "ok", log_path


# Le kernel Jupyter ne doit garder ni modele ni dataset.
for variable in (
    "model", "base_model", "finetuned_model", "baseline_model", "adapter_model",
    "st_model", "benchmark", "task", "results", "train_ds", "eval_ds",
    "train_datasets", "fr_sts_splits", "fr_nli_raw", "replay_raw",
):
    globals().pop(variable, None)
import gc
gc.collect()
if os.name == "nt":
    ctypes.windll.psapi.EmptyWorkingSet(ctypes.windll.kernel32.GetCurrentProcess())

completed = [task for task in SAFE_TASKS if valid_result(task)]
pending = [task for task in SAFE_TASKS if not valid_result(task)]
print({
    "benchmark": BENCHMARK_LABEL,
    "model_revision": MODEL_REVISION,
    "completed": len(completed),
    "pending": len(pending),
    "worker": str(WORKER_PATH),
})

for index, task_name in enumerate(pending, start=1):
    primary_device = "cpu" if CPU_ONLY else "cuda"
    print(
        f"[{index}/{len(pending)}] {task_name} - worker {primary_device.upper()} isole",
        flush=True,
    )
    success, reason, log_path = run_worker(task_name, primary_device)

    if not success and RETRY_ON_CPU and primary_device != "cpu":
        print(f"    GPU interrompu: {reason}; nouvelle tentative CPU", flush=True)
        time.sleep(5)
        success, reason, log_path = run_worker(task_name, "cpu")

    if success:
        data = json.loads(result_path(task_name).read_text(encoding="utf-8"))
        precision = data.get("precision", "?")
        print(f"    OK {data['score']:.4f} ({data['eval_device']}, {precision})", flush=True)
        error_path(task_name).unlink(missing_ok=True)
    else:
        payload = {
            "run_name": RUN_NAME,
            "task": task_name,
            "status": "failed_safely",
            "reason": reason,
            "log": str(log_path),
        }
        atomic_json(error_path(task_name), payload)
        print(f"    ECHEC SUR: {reason}; voir {log_path}", flush=True)

    current_completed = [task for task in SAFE_TASKS if valid_result(task)]
    atomic_json(
        OUTPUT_FOLDER / "run_status.json",
        {
            "run_name": RUN_NAME,
            "benchmark": BENCHMARK_LABEL,
            "completed": current_completed,
            "remaining": [task for task in SAFE_TASKS if task not in current_completed],
            "last_task": task_name,
        },
    )
    time.sleep(3)

all_scores = {}
for task_name in SAFE_TASKS:
    if valid_result(task_name):
        data = json.loads(result_path(task_name).read_text(encoding="utf-8"))
        all_scores[task_name] = float(data["score"])

print("\n" + "=" * 70)
if all_scores:
    df = pd.DataFrame(
        [{"task": task, "score": score} for task, score in all_scores.items()]
    ).sort_values("score", ascending=False).reset_index(drop=True)
    mean_score = float(df["score"].mean())
    df["score"] = df["score"].round(4)
    display(df)
    df.to_csv(OUTPUT_FOLDER / "summary.csv", index=False)
    atomic_json(
        OUTPUT_FOLDER / "summary.json",
        {
            "run_name": RUN_NAME,
            "model_label": MODEL_LABEL,
            "model_revision": MODEL_REVISION,
            "benchmark_label": BENCHMARK_LABEL,
            "official_mteb_fra_v1": False,
            "gpu_eval_mode": GPU_EVAL_MODE,
            "gpu_batch_size": GPU_BATCH_SIZE,
            "gpu_chunk_size": GPU_CHUNK_SIZE,
            "mean_score": mean_score,
            "completed_tasks": len(all_scores),
            "total_tasks": len(SAFE_TASKS),
        },
    )
    print(
        f"Score moyen {BENCHMARK_LABEL}: {mean_score:.4f} "
        f"({len(all_scores)}/{len(SAFE_TASKS)} taches)"
    )
    print(f"Resultats: {OUTPUT_FOLDER}")
else:
    print("Aucun score valide.")


{'benchmark': 'MTEB(fra)-safe v2 CPU-only', 'model_revision': 'lora-5701562d9417', 'completed': 14, 'pending': 11, 'worker': 'F:\\codeAI\\llm\\emebedding\\mteb_safe_worker.py'}
[1/11] AlloprofReranking - worker CPU isole
    0.5 min, worker 0.6 Go, RAM libre 3.8 Go
    1.0 min, worker 0.8 Go, RAM libre 3.6 Go
    1.5 min, worker 1.7 Go, RAM libre 2.9 Go
    2.0 min, worker 1.7 Go, RAM libre 2.9 Go
    2.5 min, worker 1.7 Go, RAM libre 2.9 Go
    3.0 min, worker 1.8 Go, RAM libre 2.9 Go
    3.5 min, worker 1.8 Go, RAM libre 2.9 Go
    4.0 min, worker 1.8 Go, RAM libre 2.8 Go
    4.5 min, worker 1.8 Go, RAM libre 2.8 Go
    5.0 min, worker 1.8 Go, RAM libre 2.8 Go
    5.5 min, worker 1.8 Go, RAM libre 2.7 Go
    6.0 min, worker 1.8 Go, RAM libre 3.3 Go
    6.5 min, worker 1.8 Go, RAM libre 3.3 Go
    7.0 min, worker 1.8 Go, RAM libre 3.3 Go
    7.5 min, worker 1.8 Go, RAM libre 3.3 Go
    8.0 min, worker 1.8 Go, RAM libre 3.3 Go
    8.5 min, worker 1.8 Go, RAM libre 3.3 Go
    9.0 min, w

,task,score
0,STSBenchmarkMultilingualSTS,0.8302
1,MTOPDomainClassification,0.8025
2,SICKFr,0.7744
3,MasakhaNEWSClassification,0.7659
4,MassiveScenarioClassification,0.6482
5,STS22,0.6187
6,MassiveIntentClassification,0.6098
7,PawsXPairClassification,0.5883
8,SyntecReranking,0.5442
9,MTOPIntentClassification,0.5244


Score moyen MTEB(fra)-safe v2 CPU-only: 0.4696 (25/25 taches)
Resultats: f:\codeAI\llm\emebedding\mteb_fra_results\e5_fr_multitask_lora-5701562d9417


In [2]:
import pandas as pd

df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "summary.csv"))
print(f"Score moyen ({BENCHMARK_LABEL}) : {df['score'].mean():.4f}")
print(f"Nb tâches   : {len(df)}")

# Par catégorie
categories = {
    "Classification" : ["MTOPDomainClassification", "MTOPIntentClassification", "MassiveScenarioClassification", "MassiveIntentClassification", "MasakhaNEWSClassification", "AmazonReviewsClassification"],
    "STS"            : ["STSBenchmarkMultilingualSTS", "SICKFr", "STS22"],
    "Reranking"      : ["SyntecReranking", "AlloprofReranking"],
    "PairClass."     : ["PawsXPairClassification"],
    "Clustering"     : ["AlloProfClusteringP2P", "AlloProfClusteringS2S", "MLSUMClusteringP2P.v2", "MLSUMClusteringS2S.v2", "MasakhaNEWSClusteringS2S", "MasakhaNEWSClusteringP2P", "HALClusteringS2S.v2"],
    "Retrieval"      : ["SyntecRetrieval", "XPQARetrieval", "AlloprofRetrieval", "MintakaRetrieval", "BSARDRetrieval"],
    "Summarization"  : ["SummEvalFr"],
}

for cat, tasks in categories.items():
    scores = df[df["task"].isin(tasks)]["score"]
    print(f"{cat:<15} : {scores.mean():.4f}  ({len(scores)} tâches)")


Score moyen (MTEB(fra)-safe v2 CPU-only) : 0.4696
Nb tâches   : 25
Classification  : 0.6301  (6 tâches)
STS             : 0.7411  (3 tâches)
Reranking       : 0.5260  (2 tâches)
PairClass.      : 0.5883  (1 tâches)
Clustering      : 0.3750  (7 tâches)
Retrieval       : 0.2541  (4 tâches)
Summarization   : 0.2921  (1 tâches)


In [ ]:
# Publication optionnelle (à exécuter seulement après validation MTEB) :
# finetuned_model.push_to_hub("AurelPx/multilingual-e5-base-miracl-fr-lora")